# 10 · Test Quality & CI — Coverage vs. Mutation Testing

**Focus:** why 100% line coverage can still be a lie, and how mutation testing exposes weak tests.

**Coverage** measures which *lines* your tests execute. It's useful but shallow: a line can be
*executed* without its behavior being *verified*. You can hit 100% coverage with assertions that
barely check anything.

**Mutation testing** is the honest scorecard. A tool like **`mutmut`** makes tiny changes ("mutants")
to your code — flip `>=` to `>`, change `+` to `-`, replace a constant — then reruns your tests for
each mutant. If your tests **fail** on the mutated code, they "killed" the mutant (good — they detected
the change). If they still **pass**, the mutant **survived** — proof your tests don't actually check
that behavior.

We'll build a function with **100% line coverage but a surviving mutant**, then strengthen the test to
kill it. Everything runs inside an isolated `nb10/` folder so mutmut only sees these files.

### Setup — point the shell at this course's tools
The `!` cells below run command-line tools (`pytest`, later `mutmut`, `playwright`). We prepend the
active kernel's `bin/` directory to `PATH` so those commands resolve to the versions installed for
**this course**, not some other Python on your machine. Run this cell first.

In [1]:
import sys, os
# The kernel's own interpreter lives in the course virtualenv's bin/ directory.
os.environ["PATH"] = os.path.dirname(sys.executable) + os.pathsep + os.environ["PATH"]
print("Using Python:", sys.executable)
!pytest --version

Using Python: /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson/.venv/bin/python


pytest 9.1.1


### Set up an isolated working directory
mutmut mutates *every* source file it's pointed at, so we give it its own folder.

In [2]:
import os
os.makedirs("nb10", exist_ok=True)
os.chdir("nb10")
print("working dir:", os.getcwd())

working dir: /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson/nb10


### The code under test
A grading function. The boundary is `>= 60`.

In [3]:
%%writefile grading.py
def get_grade(score):
    if score >= 60:
        return "pass"
    return "fail"

Writing grading.py


### A test with 100% line coverage — but a blind spot
It checks a clearly-passing score (80) and a clearly-failing score (50). Every line runs... but the
**boundary itself (60)** is never tested.

In [4]:
%%writefile test_grading.py
from grading import get_grade

def test_pass():
    assert get_grade(80) == "pass"

def test_fail():
    assert get_grade(50) == "fail"

Writing test_grading.py


### mutmut needs a config
Tell mutmut which file to mutate. (mutmut 3.x reads this before it does anything.)

In [5]:
%%writefile setup.cfg
[mutmut]
source_paths=grading.py

Writing setup.cfg


### CLI Demo 1 — coverage says 100% ✅
Every line of `grading.py` is executed by the tests.

In [6]:
!pytest --cov=grading --cov-report=term -q test_grading.py

..                                                                       [100%]
================================ tests coverage ================================
______________ coverage: platform darwin, python 3.12.11-final-0 _______________

Name         Stmts   Miss  Cover
--------------------------------
grading.py       4      0   100%
--------------------------------
TOTAL            4      0   100%
2 passed in 0.05s


### CLI Demo 2 — mutation testing says otherwise ❌
`mutmut run` mutates `grading.py` and reruns the tests per mutant. Watch the summary: some mutants
**survive** (the 🙁 count). 100% coverage, yet the tests miss real behavior changes.

In [7]:
!mutmut run

⠋ Generating mutants

⠹ Generating mutants
    done in 39ms (1 files mutated, 0 ignored, 0 unmodified)
⠙ Running stats     

⠧ Running stats


    done
⠼ Running clean tests


    done
⠴ Running forced fail test

⠋ Running forced fail test


    done
Running mutation testing
⠦ 0/6  🎉 0 🫥 0  ⏰ 0  🤔 0  🙁 0  🔇 0  🧙 0

⠧ 6/6  🎉 4 🫥 0  ⏰ 0  🤔 0  🙁 2  🔇 0  🧙 0
65.59 mutations/second


### Which mutants survived?
`mutmut results` lists them; `mutmut show <id>` prints the exact code change that slipped through.

In [8]:
!mutmut results

    grading.x_get_grade__mutmut_1: survived
    grading.x_get_grade__mutmut_2: survived


In [9]:
!mutmut show grading.x_get_grade__mutmut_1

# grading.x_get_grade__mutmut_1: survived
--- grading.py
+++ grading.py
@@ -1,4 +1,4 @@
 def get_grade(score):
-    if score >= 60:
+    if score > 60:
         return "pass"
     return "fail"


See it? mutmut changed `score >= 60` to `score > 60`. Our tests (80 and 50) can't tell the difference —
both operators agree except **exactly at 60**. That's the untested boundary.

### 🔬 Try it yourself — inspect the *other* survivor
We saw `mutmut_1` was `>=` → `>`. But **two** mutants survived. **Predict** what the second one changed,
then run the cell to reveal it. (Hint: think about the boundary constant `60`.)

In [10]:
!mutmut show grading.x_get_grade__mutmut_2

# grading.x_get_grade__mutmut_2: survived
--- grading.py
+++ grading.py
@@ -1,4 +1,4 @@
 def get_grade(score):
-    if score >= 60:
+    if score >= 61:
         return "pass"
     return "fail"


It flipped `>= 60` into `>= 61` — a *different* mutation, but the **same** blind spot: neither survivor
can be caught unless a test pins the exact boundary. Watch how the single `test_boundary` we add next
kills **both** of them at once.

### The fix — test the boundary
Add an assertion at the exact edge. Now `>=` and `>` disagree (60 passes under `>=`, fails under `>`),
so the mutant gets killed.

In [11]:
%%writefile test_grading.py
from grading import get_grade

def test_pass():
    assert get_grade(80) == "pass"

def test_fail():
    assert get_grade(50) == "fail"

def test_boundary():
    # The crucial case: exactly 60 must pass. This kills the >= -> > mutant.
    assert get_grade(60) == "pass"

Overwriting test_grading.py


### Re-run mutation testing — mutants killed 🎉
With the boundary covered, the surviving mutants die. This is a *stronger* suite than 100% coverage alone implied.

In [12]:
!mutmut run

⠹ Generating mutants
    done in 24ms (0 files mutated, 0 ignored, 1 unmodified)
⠙ Listing all tests 

⠋ Listing all tests


Found 1 new tests, rerunning stats collection
⠙ Running stats    

⠦ Running stats


    done
⠏ Running clean tests

⠼ Running clean tests
    done


⠋ Running forced fail test


    done
Running mutation testing
⠦ 0/6  🎉 0 🫥 0  ⏰ 0  🤔 0  🙁 0  🔇 0  🧙 0

⠧ 6/6  🎉 6 🫥 0  ⏰ 0  🤔 0  🙁 0  🔇 0  🧙 0
72.39 mutations/second


### ⚠️ Common pitfall — don't chase a 100% mutation score
Some mutants are **equivalent** — they change the code without changing its behavior, so *no* test can
kill them (e.g. `a < b` vs `a <= b` on values that never hit the boundary). Mutation testing is a
spotlight for *weak assertions*, not a metric to max out. Run it on your critical logic, kill the
meaningful survivors, and don't burn hours on equivalent ones.

### What you learned
- **Line coverage ≠ test quality.** 100% coverage can hide assertions that verify almost nothing.
- **Mutation testing** perturbs your code and checks whether tests notice — surviving mutants reveal
  real gaps.
- `mutmut run` → `mutmut results` → `mutmut show <id>` is the loop: run, list survivors, inspect the
  change, then add the missing assertion to kill it.
- In CI, combine both: coverage as a cheap floor, mutation testing (even on critical modules only) as
  the real quality bar.

**Course complete.** You've covered assertions, fixtures, parametrization, mocking, time control,
errors & logs, async, integration + snapshots, E2E browser tests, and test-quality analysis. 🎓